# Extract Achievements Data

Extracts Achievements assets and their data (description, points) from Anno 117
into CSV and JSON files. Output goes to `results/tables/`.

Run all cells from the project root.

In [1]:
# Misc: This allow reloading .py modules into jupyter.
%load_ext autoreload
%autoreload 3

## Load assets

Setup game data for processing.

In [2]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache

config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Total assets: {len(assets.elements)}")
print(f"Total texts: {len(assets.texts.elements)}")

Total assets: 32911
Total texts: 34736


## Extract Achievements assets

Extract the patrons from `Achievements` and `AchievementSet` in order to perform the ETL process.

In [3]:
from assetextractor.conversion.statistics.achievement_extractor import AchievementExtractor

extractor = AchievementExtractor(assets)
# extractor_de = AchievementExtractor(assets, language="german")
achievement_data = extractor.extract_all()
print(f"Processed {len(achievement_data.keys())} achievements")

Processed 13 achievements


## Print Achievement Sets and their details

Use `extractor.print_achievement_sets()` to see everything, or `extractor.print_achievement_sets(guid=157242)` (Set13 - Hippodrome, as example) to inspect a specific set.

In [4]:
extractor.print_achievement_sets(guid=157242)


                         ACHIEVEMENT SET: Set13 (Hippodrome) (GUID: 157242)                         
Contains 9 Achievements:

  [ACHIVEMENT 0] Achievement_Set13_01_ConstructHippodrome (GUID: 157241)
      * Circus Magnificus (GUID: 157241 | Achievement_Set13_01_ConstructHippodrome)
        Difficulty: Bronze
        Description: Complete the construction of the Hippodrome.

  [ACHIVEMENT 1] Achievement_Set13_02_FirstBet (GUID: 157243)
      * To The Victor... (GUID: 157243 | Achievement_Set13_02_FirstBet)
        Difficulty: Bronze
        Description: Accept a Rival's challenge at the Hippodrome.

  [ACHIVEMENT 2] Achievement_Set13_03_FirstWin (GUID: 157244)
      * ...The Spoils (GUID: 157244 | Achievement_Set13_03_FirstWin)
        Difficulty: Bronze
        Description: Win a race with one of your specialists.

  [ACHIVEMENT 3] Achievement_Set13_04_MaxRacerStats (GUID: 157245)
      * Per Ardua Ad Alta (GUID: 157245 | Achievement_Set13_04_MaxRacerStats)
        Difficulty: Bronze

In [7]:
from pathlib import Path

# Define base output folder
output_root = Path("results")

# =================================================================
# SCENARIO A: MIRRORED STRUCTURE (Mimics Game Folders)
# Use this for: Webapps that need the full "base/icon_content/..." tree.
# =================================================================
print(f" Scenario A: Mirrored ".center(80, "*"))

# 1. Export ALL physical files (Icons @ 128px, Big Portraits @ 512px, Small Portraits @ 128px)
# Recreates the game's directory hierarchy for everything.
extractor.export_all_assets(
    output_base=output_root / "icons",
    quality=75,
    flatten=False        # Mirror hierarchy for standard assets & portraits
)

# 2. Save JSON with URLs like: "base/icon_content/religion/icon_name.webp"
extractor.save_to_json(
    file_path=output_root / "tables/achievements_en.json",
    web_base_path="assets/icons",  # No prefix (None) -> relative path from game tree root
    flatten=False             # Matches the mirrored file export paths
)

***************************** Scenario A: Mirrored *****************************
Started exporting 117 achievement icons...
  [OK] -> icons\base\icon_content\achievements\achievement_set01_01_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_03_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_02_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_04_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_05_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_06_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_07_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_08_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set01_09_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set02_01_0.webp
  [OK] -> icons\base\icon_content\achievements\achievement_set02_02_0.webp
  [OK] -> icons\base\icon_content\achievements\achi

## Display as tables

In [6]:
from IPython.display import display, HTML

# Define your color palette
HEADER_BG = "#5F032E"
ROW_BG = "#EBD2B8"
TEXT_COLOR = "#1D000E"

for set_name, set_df in dfs_by_set.items():
    display(HTML(f"<h2>Set: {set_name}</h2>"))
    
    # 1. Clean the dataframe for display
    display_df = set_df.drop(columns=["Set Name", "Set UID"])
    
    # 2. Apply styling
    styled_df = (display_df.style
        # Set overall row background and text color
        .set_properties(**{
            'background-color': ROW_BG,
            'color': TEXT_COLOR,
            'border': f'1px solid {HEADER_BG}',
            'padding': '8px'
        })
        # Set header background and text color
        .set_table_styles([
            {
                'selector': 'th',
                'props': [
                    ('background-color', HEADER_BG),
                    ('color', 'white'),
                    ('font-weight', 'bold'),
                    ('text-align', 'center')
                ]
            }
        ])
        .hide(axis="index") # Remove the index column
    )
    
    display(styled_df)

NameError: name 'dfs_by_set' is not defined

## Export JSON

Nested dict per AchievementSet per Achievement, with the first level key being the set GUID and the second level for each achievement inside `achivements` dictionary being the achievement uid.

In [ ]:
def clean_icon_path(raw_path: str | None) -> str | None:
    """Removes everything before and including '.cache' and strips extension."""
    if not raw_path:
        return None
    
    # Use partition to split at '.cache'
    # .partition returns (before, separator, after)
    _, sep, after = raw_path.partition(".cache")
    
    if sep:
        # Remove the file extension (e.g., .dds)
        return str(Path(after).with_suffix(""))
    
    return raw_path

def export_nested_achievements(sets_dict: dict[str, AchievementSet], output_path):
    final_json = {}

    for set_uid, set_data in sets_dict.items():
        # Level 1: Keyed by SetUID
        final_json[str(set_uid)] = {
            "category": set_data["category"],
            "reward": set_data["reward"],  # This is already your I18N dict
            "achievements": {}
        }

        # Level 2: Keyed by AchievementUID inside "achievements"
        for ach in set_data["achievements"]:
            ach_uid = str(ach["uid"])

            cleaned_path = clean_icon_path(ach["icon"]["path"])
            
            final_json[str(set_uid)]["achievements"][ach_uid] = {
                "title": ach["title"],             # Full I18N dict
                "description": ach["description"], # Full I18N dict
                "points": int(ach["points"]),      # Cast to standard int
                "difficulty": ach["difficulty"],
                "image_url": cleaned_path   # Using the path from icon data
            }

    # Write with ensure_ascii=False to keep localizations readable
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_json, f, indent=4)

# Execution
target_json = output_dir / "achievements.json"
export_nested_achievements(achievement_sets_dict, target_json)

print(f"Successfully exported nested JSON to {target_json}")

## Image Export

Convert every achievement image from .DDS to .webp using wand + magick

In [ ]:
from wand.image import Image
import os

# 1. Flatten all icons from all sets into a single list
# We use a set for 'seen_paths' to ensure icons_data only contains unique entries
icons_data: dict[str, IconData] = {}
seen_paths = set()

for set_data in achievement_sets_dict.values():
    for ach in set_data["achievements"]:
        icon = ach["icon"]
        path = icon.get("path")
        
        if path and path not in seen_paths:
            icons_data[ach["uid"]] = ach["icon"]
            seen_paths.add(path)

print(f"Extracted {len(icons_data)} unique icons.")

## Start the conversion and export logic.
print("Started Image Export")

for guid, data in icons_data.items():
    img = data["image"]
    original_path = data["path"]
    
    # Skip if there is no image or path associated with this GUID
    if not img or not original_path:
        continue
    
    # Construct output path (swap .dds for .webp)
    output_path = os.path.splitext(original_path)[0] + ".webp"
    
    # print(f"Exporting Achievement Image - GUID: {guid}")
    # print(f"  Source: {os.path.basename(original_path)}")
    # print(f"  Target: {output_path}")
    
    try:
        # We use the existing WandImageProto object
        # Ensure format is set to webp for the encoder
        img.format = 'webp'
        img.save(filename=output_path)
    except Exception as e:
        print(f"  [ERROR] Failed to export {guid}: {e}")
    
print("---")
print("Finished Exporting")